In [ ]:
import time
import datetime as dt
import requests
import pandas as pd
import os
from pathlib import Path
from dotenv import load_dotenv

def _find_project_root(marker="requirements.txt"):
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate project root (missing {marker})")

PROJECT_ROOT = _find_project_root()
DATA_DIR = PROJECT_ROOT / "O" / "U" / "Data"

load_dotenv(PROJECT_ROOT / ".env")
API_KEY = os.environ["ODDS_API_KEY"]
SPORT = "baseball_mlb"
REGION = "us"                           # single region keeps credit cost down
MARKET = "totals"                       # over/under only — this is what we need
BASE = "https://api.the-odds-api.com/v4"
 
 
def fetch_daily_snapshot(date_iso: str) -> dict | None:
    """
    Pull one historical snapshot of all MLB totals at the given timestamp.
    Returns the parsed JSON (a dict with 'timestamp' and 'data'), or None on failure.
    """
    url = f"{BASE}/historical/sports/{SPORT}/odds"
    params = {
        "apiKey": API_KEY,
        "regions": REGION,
        "markets": MARKET,
        "oddsFormat": "american",
        "date": date_iso,
    }
    resp = requests.get(url, params=params, timeout=30)
 
    # The API returns remaining-credits in headers — worth watching
    remaining = resp.headers.get("x-requests-remaining")
    used = resp.headers.get("x-requests-used")
    print(f"  {date_iso[:10]} -> status {resp.status_code} | "
          f"credits used={used}, remaining={remaining}")
 
    if resp.status_code != 200:
        print(f"  WARNING: non-200 for {date_iso}: {resp.text[:200]}")
        return None
    return resp.json()
 
 
def flatten_snapshot(snapshot: dict) -> list[dict]:
    """
    Turn one snapshot's nested JSON into flat rows:
    one row per (game, bookmaker) with the totals line + over/under prices.
    """
    rows = []
    snap_ts = snapshot.get("timestamp")
    games = snapshot.get("data", [])
 
    for game in games:
        base = {
            "snapshot_ts": snap_ts,
            "event_id": game["id"],
            "commence_time": game["commence_time"],
            "home_team": game["home_team"],
            "away_team": game["away_team"],
        }
        for book in game.get("bookmakers", []):
            for market in book.get("markets", []):
                if market["key"] != "totals":
                    continue
                # totals market has two outcomes: Over and Under, each with a 'point'
                over = next((o for o in market["outcomes"] if o["name"] == "Over"), None)
                under = next((o for o in market["outcomes"] if o["name"] == "Under"), None)
                rows.append({
                    **base,
                    "bookmaker": book["key"],
                    "line_last_update": market.get("last_update"),
                    "total_line": over["point"] if over else None,
                    "over_price": over["price"] if over else None,
                    "under_price": under["price"] if under else None,
                })
    return rows
 
 
def pull_season(start_date: dt.date, end_date: dt.date,
                snapshot_hour_utc: int = 16) -> pd.DataFrame:
    """
    Loop day-by-day over the range, pull one snapshot each, flatten, concat.
    snapshot_hour_utc: hour of day (UTC) to grab the snapshot — 16:00 UTC is
    a reasonable 'morning of' capture for US games.
    """
    all_rows = []
    day = start_date
    while day <= end_date:
        ts = f"{day.isoformat()}T{snapshot_hour_utc:02d}:00:00Z"
        snap = fetch_daily_snapshot(ts)
        if snap:
            all_rows.extend(flatten_snapshot(snap))
        time.sleep(0.5)   # be polite; avoid hammering the API
        day += dt.timedelta(days=1)
 
    df = pd.DataFrame(all_rows)
    return df
 
 
if __name__ == "__main__":
    # --- pull 2025 regular season (adjust dates as needed) ---
    df = pull_season(dt.date(2026, 3, 27), dt.date(2026, 7, 1))
 
    print(f"\nPulled {len(df)} rows across "
          f"{df['event_id'].nunique() if not df.empty else 0} unique games")
 
    out_path = DATA_DIR / "mlb_totals_2026_raw.csv"
    df.to_csv(out_path, index=False)
    print(f"Saved -> {out_path}")